# 9장 · 그래디언트와 경사하강법 ⛷️

웹 챕터 **"그래디언트와 경사하강법"** 과 짝을 이루는 Python 실습입니다.
경사하강 루프를 직접 코딩해 w 가 최소값으로 수렴하는 과정을 보고, 학습률을 바꿔
진동·발산도 재현합니다. (AI 학습의 심장!)

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch9')

## 1. 경사하강 루프: w ← w − η·f'(w)

손실 f(w)=w² (기울기 2w) 에서 w=5 부터 시작해 골짜기 바닥(0)으로 내려갑니다.

- **η(에타) = 학습률**: 한 걸음에 얼마나 크게 내려갈지 정하는 보폭.
- 기울기 f'(w) 는 가장 가파른 **오르막** 방향이라, **부호를 빼서(−)** 내리막으로 이동합니다.

In [ ]:
def f(w):
    return w**2
def df(w):
    return 2*w

w = 5.0
eta = 0.1
history = [w]
for i in range(15):
    w = w - eta * df(w)        # 갱신식
    history.append(w)
    print(f'{i+1:2d}걸음: w = {w:.4f},  손실 = {f(w):.4f}')
print('→ 0 으로 수렴!')

## 2. 내려가는 자취 그리기

손실 곡선 위에 매 걸음의 위치를 찍어, 공이 골짜기로 굴러 내려가는 모습을 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

ws = np.linspace(-5.5, 5.5, 200)
hist = np.array(history)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(ws, f(ws), color='#6366f1', lw=2.5, label='손실 f(w)=w²')
ax.plot(hist, f(hist), 'o-', color='#ef4444', ms=6, lw=1, label='경사하강 자취')
ax.scatter([hist[0]], [f(hist[0])], color='black', zorder=5, label='시작')
ax.grid(True, color='#eef2f7'); ax.legend()
ax.set_title('공이 골짜기로 굴러 내려간다 (η=0.1)')
plt.show()

## 3. 학습률 실험: 작게 / 적당 / 너무 크게

같은 시작점에서 η 만 바꿔 봅니다. η 가 크면 최소값을 지나쳐 진동·발산해요.

In [ ]:
def run(eta, steps=12, w0=5.0):
    w = w0
    hist = [w]
    for _ in range(steps):
        w = w - eta * df(w)
        hist.append(w)
    return hist

for eta in [0.02, 0.1, 0.5, 1.05]:
    h = run(eta)
    tag = '수렴' if abs(h[-1]) < abs(h[0]) else '발산'
    print(f'η={eta:<5}  마지막 w = {h[-1]:+.3f}   ({tag})')

## 4. 2D 경사하강: 그래디언트 벡터로

f(x,y)=x²+y² 의 그래디언트 ∇f=(2x, 2y) 를 따라 (x,y) 가 원점(최소)으로 수렴합니다.

In [ ]:
def grad(x, y):
    return np.array([2*x, 2*y])   # ∇f = (∂f/∂x, ∂f/∂y)

p = np.array([4.0, 3.0])
eta = 0.15
print('시작:', p)
for i in range(10):
    p = p - eta * grad(p[0], p[1])   # 벡터 갱신: p ← p − η·∇f
print('10걸음 뒤:', np.round(p, 4), ' (원점 0,0 으로 수렴)')

## 5. 직접 풀어보기 ✏️

f(w)=w², η=0.1, w=5 에서 **한 걸음 뒤의 w** 를 구해 보세요. (웹 문제와 같습니다. 답은 4!)

In [ ]:
# TODO: 갱신식 w − η·df(w) 로 한 걸음 계산해 answer 에 담으세요.
# 힌트: 5 - 0.1 * df(5)

answer = 5 - 0.1 * df(5)
print('정답:', answer)
assert abs(answer - 4) < 1e-9, '다시! 기울기 2·5=10, 5 − 0.1×10 = 4.'
print('✅ 잘했어요! w: 5 → 4. 기울기 반대 방향으로 η 만큼 이동.')

---
마지막 장(10장): 이 ∇L 을 신경망에서 **효율적으로** 구하는 방법인 **역전파**로
지금까지 배운 모든 미분(연쇄법칙·편미분·활성화 미분)을 하나로 잇습니다. 🙂